In [ ]:
from google.colab import files
files.upload()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv('p1_iris.csv')
d = df[df.species != 'setosa'].reset_index(drop=True)
X = d[['petal_length', 'petal_width']].values
y = (d.species == 'virginica').astype(int).values

def fit(X, y):
    best = (0, 0, 0)
    for j in range(X.shape[1]):
        for t in np.unique(X[:, j]):
            acc = ((X[:, j] > t) == y).mean()
            if acc > best[0]:
                best = (acc, j, t)
    return best[1], best[2]

def predict(X, j, t):
    return (X[:, j] > t).astype(int)

def scores(y, p):
    return [accuracy_score(y, p), precision_score(y, p, zero_division=0), recall_score(y, p, zero_division=0), f1_score(y, p)]

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
j, t = fit(Xtr, ytr)
res = {'Train/Test split': scores(yte, predict(Xte, j, t))}
cv = []
for a, b in StratifiedKFold(3, shuffle=True, random_state=42).split(X, y):
    jj, tt = fit(X[a], y[a])
    cv.append(scores(y[b], predict(X[b], jj, tt)))
res['3-Fold CV (mean)'] = np.mean(cv, axis=0)
print(pd.DataFrame(res, index=['Accuracy', 'Precision', 'Recall', 'F1']).T.round(3))

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k')
if j == 0:
    plt.axvline(t, color='k')
else:
    plt.axhline(t, color='k')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.title('Manual decision boundary')
plt.show()